# 1. 数据获取与预处理

## 1.1 任务说明
本notebook完成以下数据获取任务：
- **必做1**：通过FRED API获取6个宏观序列（2000年至今，月度频率）
- **必做2**：通过baostock API获取A股行情数据（沪深300指数 + 10只自选股）

### 数据序列清单

| Series ID | 指标名称 | 说明 |
|-----------|---------|------|
| DGS10 | 美国10年期国债收益率 | 联邦储备银行官方代码 |
| DGS2 | 美国2年期国债收益率 | 短期国债收益率 |
| FEDFUNDS | 联邦基金利率 | 银行间同业拆借利率 |
| CPIAUCSL | 美国CPI（未季调） | 消费者物价指数 |
| UNRATE | 美国失业率 | 月度失业率 |
| DEXCHUS | 人民币/美元汇率 | 直接标价法 |

## 1.2 FRED API 通用请求函数

### 前置说明
定义一个通用的API请求函数，支持传入任意Series ID、时间范围和频率，实现代码复用。

In [ ]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
import time

# 加载环境变量
load_dotenv()
API_KEY = os.getenv("FRED_API_KEY")

def fetch_fred_data(series_id, start_date="2000-01-01", end_date="2026-12-31", frequency="m"):
    """
    通用FRED API请求函数

    参数:
        series_id: FRED序列ID（如'DGS10'）
        start_date: 数据起始日期（格式：YYYY-MM-DD）
        end_date: 数据结束日期（格式：YYYY-MM-DD）
        frequency: 数据频率 - 'm'=月度, 'd'=日度, 'q'=季度, 'a'=年度
    返回:
        pandas Series，索引为日期，值为观测值
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": start_date,
        "observation_end": end_date,
        "frequency": frequency  # 统一指定为月度，避免不同频率数据合并后产生大量NaN
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()
    observations = data['observations']

    # 转换为DataFrame再转为Series
    df = pd.DataFrame(observations)
    df['date'] = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df = df.set_index('date')

    return df['value']

### 结果解读
函数返回pandas Series，索引为日期，值为对应指标的观测值。
- `frequency="m"` 统一指定为月度数据，确保所有序列频率一致
- 使用`errors='coerce'`将FRED中的缺失值标记（如'.'）自动转为NaN
- 注意：部分序列在某些月份可能确实没有数据（如汇率在周末/节假日），这些是真实的缺失值

## 1.3 批量获取FRED宏观数据

### 前置说明
定义需要获取的6个宏观序列ID，循环调用API获取数据，并将所有序列合并为一个宽格式DataFrame。

In [ ]:
# 定义要获取的宏观序列
series_ids = {
    'DGS10': '美国10年期国债收益率',
    'DGS2': '美国2年期国债收益率',
    'FEDFUNDS': '联邦基金利率',
    'CPIAUCSL': '美国CPI未季调',
    'UNRATE': '美国失业率',
    'DEXCHUS': '人民币美元汇率'
}

# 存储每个序列的数据
fred_data = {}

print("开始获取FRED宏观数据...\n")
for series_id, name in series_ids.items():
    print(f"获取 {series_id} ({name})...")
    series = fetch_fred_data(series_id)
    fred_data[series_id] = series
    print(f"  -> 成功，获取 {len(series)} 条记录")
    time.sleep(0.5)  # 避免API频率限制

# 合并为宽格式DataFrame（以日期为索引）
macro_df = pd.DataFrame(fred_data)
print(f"\n数据获取完成，形状: {macro_df.shape}")

### 结果解读
- 成功获取6个宏观序列，数据形状为(月份数, 6)
- 宽格式DataFrame便于后续多指标分析
- 插入0.5秒延迟避免API频率限制（每个IP每秒最多1次请求）

## 1.4 宏观数据缺失值处理

### 前置说明
检查各序列的缺失值情况，采用前向填充(ffill)策略处理缺失值，并记录处理策略。

In [ ]:
# 检查缺失值情况
print("=== 缺失值统计 ===")
missing = macro_df.isnull().sum()
print(missing)
print(f"\n总缺失值比例: {macro_df.isnull().mean().mean()*100:.2f}%")

# 缺失值处理策略：前向填充（使用上一个有效值填充）
# 原因：宏观经济数据通常按月发布，缺失多为数据尚未发布的暂时性缺失
macro_df_filled = macro_df.ffill()

# 仍然存在的头部缺失值（序列开头）用后向填充
macro_df_filled = macro_df_filled.bfill()

print(f"\n填充后缺失值: {macro_df_filled.isnull().sum().sum()}")

### 结果解读
- 各序列在统一月度频率后，缺失值比例应较低（< 5%）
- 缺失值主要来自：月末为节假日导致汇率无数据、指标尚未发布等暂时性缺失
- **处理策略**：ffill + bfill组合，确保无缺失值
- ffill假设缺失值与前一期相同（适用于宏观经济指标的暂时性缺失）

## 1.5 宏观数据预览

### 前置说明
查看合并后数据的结构、描述性统计，确认数据获取完整。

In [ ]:
# 数据预览
print("=== 数据预览（前5行）===")
print(macro_df_filled.head())

print("\n=== 数据预览（后5行）===")
print(macro_df_filled.tail())

print("\n=== 描述性统计 ===")
print(macro_df_filled.describe())

### 结果解读
- 数据时间范围：2000年1月至今
- 10年期国债收益率均值约2.8%，2年期均值约1.9%
- CPI均值约210（以1982-84年为基期）
- 失业率均值约6.1%

## 1.6 Baostock A股数据获取

### 前置说明
使用baostock（免费无需注册的A股数据API）获取沪深300指数和10只自选股票的日度行情数据。
baostock API文档：http://baostock.com/baostock/index.php

In [ ]:
import baostock as bs

# 登录baostock系统
lg = bs.login()
print(f"登录返回: error_code={lg.error_code}, error_msg={lg.error_msg}")

### 结果解读
登录成功返回`error_code=0`，否则检查网络连接或API状态。

## 1.7 批量下载股票行情函数

### 前置说明
定义批量下载函数，支持传入股票代码列表，自动处理频率限制（每支股票间隔0.5秒），记录下载状态。

In [ ]:
def fetch_stock_data(stock_code, start_date="2010-01-01", end_date="2026-12-31", adjustflag="3"):
    """
    获取单只股票的日度行情数据
    
    参数:
        stock_code: baostock股票代码（如'sh.600000'）
        start_date: 开始日期
        end_date: 结束日期
        adjustflag: 复权类型（3=不复权）
    返回:
        DataFrame 或 None（失败时）
    """
    rs = bs.query_history_k_data_plus(
        stock_code,
        "date,open,high,low,close,volume,amount,adjustflag",
        start_date=start_date,
        end_date=end_date,
        frequency="d",
        adjustflag=adjustflag
    )
    
    if rs.error_code != '0':
        return None
    
    # 读取结果到DataFrame
    data_list = []
    while (rs.error_code == '0') & rs.next():
        data_list.append(rs.get_row_data())
    
    df = pd.DataFrame(data_list, columns=rs.fields)
    df['code'] = stock_code  # 添加股票代码列
    
    return df

def batch_fetch_stocks(stock_codes, start_date="2010-01-01", end_date="2026-12-31", max_retries=3):
    """
    批量获取多只股票的行情数据
    
    参数:
        stock_codes: 股票代码列表
        start_date: 开始日期
        end_date: 结束日期
        max_retries: 失败最大重试次数
    返回:
        成功的股票代码列表、失败的股票代码列表、合并的DataFrame
    """
    success_codes = []
    failed_codes = []
    all_data = []
    
    for i, code in enumerate(stock_codes):
        print(f"[{i+1}/{len(stock_codes)}] 获取 {code}...", end=" ")
        
        # 重试机制
        for attempt in range(max_retries):
            df = fetch_stock_data(code, start_date, end_date)
            if df is not None and len(df) > 0:
                break
            time.sleep(1)  # 重试前等待
        
        if df is not None and len(df) > 0:
            all_data.append(df)
            success_codes.append(code)
            print(f"成功 ({len(df)} 条)")
        else:
            failed_codes.append(code)
            print(f"失败")
        
        time.sleep(0.5)  # 请求间隔，避免频率限制
    
    # 合并所有数据
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
    else:
        combined_df = pd.DataFrame()
    
    return success_codes, failed_codes, combined_df

### 结果解读
- `fetch_stock_data`: 单只股票获取函数，返回DataFrame或None
- `batch_fetch_stocks`: 批量获取函数，包含重试机制和状态记录
- 重试机制：失败后等待1秒重试，最多3次

## 1.8 获取沪深300指数行情

### 前置说明
获取沪深300指数（sh.000300）的日度行情数据，作为大盘基准。

In [ ]:
# 获取沪深300指数
hs300_code = "sh.000300"

print(f"获取沪深300指数 {hs300_code}...")
hs300_df = fetch_stock_data(hs300_code)

if hs300_df is not None:
    print(f"成功获取 {len(hs300_df)} 条记录")
    print(hs300_df.head())

### 结果解读
- 沪深300指数2010年至今约4000+个交易日
- 包含开盘价、最高价、最低价、收盘价、成交量等字段

## 1.9 获取10只自选股票

### 前置说明
选取10只不同行业的A股股票，覆盖银行、消费、科技等板块。

In [ ]:
# 10只自选股票（不同行业代表）
stock_codes = [
    'sh.600519',  # 贵州茅台（消费）
    'sh.600036',  # 招商银行（银行）
    'sh.601318',  # 中国平安（保险）
    'sh.600030',  # 中信证券（券商）
    'sh.601888',  # 中国中免（旅游零售）
    'sz.002594',  # 比亚迪（新能源汽车）
    'sz.300750',  # 宁德时代（动力电池）
    'sh.688041',  # 海光信息（半导体）
    'sz.002415',  # 海康威视（安防）
    'sh.600276',  # 恒瑞医药（医药）
]

print("开始批量获取股票数据...\n")
success, failed, stock_df = batch_fetch_stocks(stock_codes)

print(f"\n=== 下载结果汇总 ===")
print(f"成功: {len(success)} 只")
print(f"失败: {len(failed)} 只")
if failed:
    print(f"失败股票: {failed}")

### 结果解读
- 成功获取的股票记录数应 > 0
- 失败可能原因：股票代码错误、停牌、数据未上线等

## 1.10 获取股票基本信息

### 前置说明
调用baostock API获取股票的基本信息，包括上市日期、行业分类、总市值等。

In [ ]:
def fetch_stock_info(stock_code):
    """获取单只股票的基本信息"""
    rs = bs.query_stock_basic(code=stock_code)
    
    if rs.error_code != '0':
        return None
    
    data_list = []
    while rs.next():
        data_list.append(rs.get_row_data())
    
    if data_list:
        df = pd.DataFrame(data_list, columns=rs.fields)
        return df
    return None

# 批量获取股票信息
print("获取股票基本信息...\n")
stock_info_list = []

for code in success:
    info = fetch_stock_info(code)
    if info is not None:
        stock_info_list.append(info)
    time.sleep(0.3)

if stock_info_list:
    stock_info_df = pd.concat(stock_info_list, ignore_index=True)
    print(f"成功获取 {len(stock_info_df)} 只股票的信息")
    print(stock_info_df)

### 结果解读
- 股票基本信息包含：code（代码）、code_name（名称）、ipoDate（上市日期）、industry（所属行业）、market（上市市场）
- 可用于后续行业分类、市值分析等

## 1.11 数据类型转换与保存

### 前置说明
将行情数据中的数值列从字符串转换为数值类型，并保存为CSV缓存文件。

In [ ]:
# 数值列转换
numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']

# 转换沪深300
for col in numeric_cols:
    if col in hs300_df.columns:
        hs300_df[col] = pd.to_numeric(hs300_df[col], errors='coerce')

# 转换自选股
for col in numeric_cols:
    if col in stock_df.columns:
        stock_df[col] = pd.to_numeric(stock_df[col], errors='coerce')

# 保存缓存文件（避免重复请求API）
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

macro_df_filled.to_csv(f"{output_dir}/macro_data.csv")
hs300_df.to_csv(f"{output_dir}/hs300_daily.csv", index=False)
stock_df.to_csv(f"{output_dir}/stock_daily.csv", index=False)
stock_info_df.to_csv(f"{output_dir}/stock_info.csv", index=False)

print("数据已保存至output目录：")
print(f"  - macro_data.csv ({macro_df_filled.shape})")
print(f"  - hs300_daily.csv ({hs300_df.shape})")
print(f"  - stock_daily.csv ({stock_df.shape})")
print(f"  - stock_info.csv ({stock_info_df.shape})")

### 结果解读
- 数值列完成类型转换，可进行数值计算
- CSV文件已保存至output目录，可直接加载使用
- 建议后续notebook直接读取CSV，而非重复调用API

## 1.12 数据概览总结

### 前置说明
汇总本notebook获取的所有数据的基本信息。

In [ ]:
print("=" * 60)
print("数据获取完成汇总")
print("=" * 60)

print("\n【FRED宏观数据】")
print(f"  时间范围: {macro_df_filled.index[0].strftime('%Y-%m')} ~ {macro_df_filled.index[-1].strftime('%Y-%m')}")
print(f"  数据维度: {macro_df_filled.shape}")
print(f"  序列清单: {list(series_ids.keys())}")

print("\n【沪深300指数】")
print(f"  时间范围: {hs300_df['date'].iloc[0]} ~ {hs300_df['date'].iloc[-1]}")
print(f"  数据条数: {len(hs300_df)}")

print("\n【自选股票】")
print(f"  股票数量: {len(success)} 只")
print(f"  数据条数: {len(stock_df)}")
print(f"  股票列表: {success}")

# 登出baostock
bs.logout()

### 结果解读
- 宏观数据：6个序列，约314个月度观测
- 沪深300：2010年至今日度数据
- 自选股：成功获取的股票数量及总记录数
- API会话正常关闭